# Diamond Price Prediction - Complete ML Pipeline

This notebook demonstrates a complete machine learning workflow including:
- Data loading and exploration
- Data cleaning and feature engineering
- Pipeline creation with column transformers
- Model comparison using cross-validation
- Hyperparameter tuning with GridSearchCV
- Final model evaluation

**Models used:** Linear Regression, Decision Tree, Random Forest, XGBoost, KNN, SVM

## 1. Import Libraries

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings('ignore')

# Preprocessing and Pipeline
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Models
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from xgboost import XGBRegressor

# Metrics
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# Set display options
pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')

print("Libraries imported successfully!")

## 2. Load and Explore Data

In [ ]:
# Load data
df = pd.read_csv('/kaggle/input/diamonds/diamonds.csv')

# Remove index column if exists
if 'Unnamed: 0' in df.columns:
    df = df.drop('Unnamed: 0', axis=1)

print(f"Dataset shape: {df.shape}")
print("\nFirst few rows:")
df.head()

In [ ]:
# Data info
print("Dataset Information:")
df.info()

In [ ]:
# Statistical summary
print("Statistical Summary:")
df.describe()

In [ ]:
# Check for missing values
print("Missing values:")
print(df.isnull().sum())

In [ ]:
# Categorical features
print("Unique values in categorical columns:")
for col in ['cut', 'color', 'clarity']:
    print(f"\n{col}: {df[col].unique()}")

## 3. Data Visualization

In [ ]:
# Price distribution
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.hist(df['price'], bins=50, edgecolor='black')
plt.xlabel('Price')
plt.ylabel('Frequency')
plt.title('Price Distribution')

plt.subplot(1, 2, 2)
plt.hist(np.log1p(df['price']), bins=50, edgecolor='black', color='orange')
plt.xlabel('Log(Price)')
plt.ylabel('Frequency')
plt.title('Log Price Distribution')

plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap (numerical features only)
plt.figure(figsize=(10, 8))
numerical_cols = df.select_dtypes(include=[np.number]).columns
correlation = df[numerical_cols].corr()
sns.heatmap(correlation, annot=True, cmap='coolwarm', center=0)
plt.title('Correlation Heatmap')
plt.show()

## 4. Data Cleaning

In [ ]:
print(f"Original shape: {df.shape}")

# Remove diamonds with zero dimensions
df = df[(df[['x', 'y', 'z']] != 0).all(axis=1)]
print(f"Shape after removing zero dimensions: {df.shape}")

# Remove outliers in depth and table
df = df[(df['depth'] >= 45) & (df['depth'] <= 75)]
df = df[(df['table'] >= 40) & (df['table'] <= 80)]
print(f"Shape after removing outliers: {df.shape}")

# Reset index
df = df.reset_index(drop=True)
print("\nData cleaning completed!")

## 5. Feature Engineering

In [ ]:
# Create volume feature
df['volume'] = df['x'] * df['y'] * df['z']

print("Feature engineering completed!")
print(f"\nFinal dataset shape: {df.shape}")
df.head()

## 6. Prepare Data for Modeling

In [ ]:
# Separate features and target
X = df.drop('price', axis=1)
y = df['price']

# Define categorical and numerical columns
categorical_features = ['cut', 'color', 'clarity']
numerical_features = ['carat', 'depth', 'table', 'x', 'y', 'z', 'volume']

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"\nCategorical features: {categorical_features}")
print(f"Numerical features: {numerical_features}")

In [ ]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")

## 7. Create Preprocessing Pipeline

In [ ]:
# Define ordinal encoding mappings for categorical features
cut_categories = ['Fair', 'Good', 'Very Good', 'Premium', 'Ideal']
color_categories = ['J', 'I', 'H', 'G', 'F', 'E', 'D']
clarity_categories = ['I1', 'SI2', 'SI1', 'VS2', 'VS1', 'VVS2', 'VVS1', 'IF']

# Create preprocessing pipeline using ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OrdinalEncoder(
            categories=[cut_categories, color_categories, clarity_categories],
            handle_unknown='use_encoded_value',
            unknown_value=-1
        ), categorical_features)
    ]
)

print("Preprocessing pipeline created successfully!")

## 8. Model Comparison with Cross-Validation

In [ ]:
# Define models
models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'Random Forest': RandomForestRegressor(random_state=42, n_jobs=-1),
    'XGBoost': XGBRegressor(random_state=42, n_jobs=-1),
    'KNN': KNeighborsRegressor(n_jobs=-1),
    'SVM': SVR()
}

print("Models initialized:")
for name in models.keys():
    print(f"  - {name}")

In [ ]:
# Perform cross-validation for each model
cv_results = []

print("Performing 5-fold cross-validation...\n")

for name, model in models.items():
    # Create pipeline
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('model', model)
    ])

    # Perform cross-validation
    cv_scores = cross_val_score(
        pipeline, X_train, y_train,
        cv=5,
        scoring='r2',
        n_jobs=-1
    )

    cv_results.append({
        'Model': name,
        'CV Mean R2': cv_scores.mean(),
        'CV Std R2': cv_scores.std(),
        'CV Scores': cv_scores
    })

    print(f"{name}:")
    print(f"  Mean R2: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")
    print(f"  Individual scores: {cv_scores}\n")

In [ ]:
# Create results dataframe
cv_results_df = pd.DataFrame(cv_results)
cv_results_df = cv_results_df.sort_values('CV Mean R2', ascending=False).reset_index(drop=True)

print("Cross-Validation Results Summary:")
print(cv_results_df[['Model', 'CV Mean R2', 'CV Std R2']].to_string(index=False))

In [ ]:
# Visualize cross-validation results
plt.figure(figsize=(12, 6))

models_list = cv_results_df['Model'].tolist()
means = cv_results_df['CV Mean R2'].tolist()
stds = cv_results_df['CV Std R2'].tolist()

plt.barh(models_list, means, xerr=stds, capsize=5)
plt.xlabel('R2 Score')
plt.ylabel('Model')
plt.title('Model Comparison - Cross-Validation Results')
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Hyperparameter Tuning with GridSearchCV

We'll tune the top 3 performing models from cross-validation.

In [ ]:
# Select top 3 models for hyperparameter tuning
top_models = cv_results_df.head(3)['Model'].tolist()
print(f"Top 3 models for hyperparameter tuning: {top_models}")

In [ ]:
# Define parameter grids for each model
param_grids = {
    'Linear Regression': {},  # No hyperparameters to tune

    'Decision Tree': {
        'model__max_depth': [10, 20, 30, None],
        'model__min_samples_split': [2, 5, 10],
        'model__min_samples_leaf': [1, 2, 4]
    },

    'Random Forest': {
        'model__n_estimators': [100, 200],
        'model__max_depth': [20, 30, None],
        'model__min_samples_split': [2, 5],
        'model__min_samples_leaf': [1, 2]
    },

    'XGBoost': {
        'model__n_estimators': [100, 200],
        'model__learning_rate': [0.01, 0.1],
        'model__max_depth': [3, 5, 7],
        'model__subsample': [0.8, 1.0]
    },

    'KNN': {
        'model__n_neighbors': [3, 5, 7, 9],
        'model__weights': ['uniform', 'distance'],
        'model__p': [1, 2]
    },

    'SVM': {
        'model__C': [0.1, 1, 10],
        'model__kernel': ['rbf', 'linear'],
        'model__gamma': ['scale', 'auto']
    }
}

print("Parameter grids defined for all models.")

In [ ]:
# Perform GridSearchCV for top models
best_models = {}
grid_results = []

print("Performing GridSearchCV for top models...\n")

for model_name in top_models:
    if model_name == 'Linear Regression':
        # No tuning needed for Linear Regression
        pipeline = Pipeline([
            ('preprocessor', preprocessor),
            ('model', LinearRegression())
        ])
        pipeline.fit(X_train, y_train)
        best_models[model_name] = pipeline
        print(f"{model_name}: No hyperparameters to tune")
        continue

    print(f"Tuning {model_name}...")

    # Create pipeline
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('model', models[model_name])
    ])

    # GridSearchCV
    grid_search = GridSearchCV(
        pipeline,
        param_grids[model_name],
        cv=3,
        scoring='r2',
        n_jobs=-1,
        verbose=1
    )

    grid_search.fit(X_train, y_train)

    best_models[model_name] = grid_search.best_estimator_

    grid_results.append({
        'Model': model_name,
        'Best Score': grid_search.best_score_,
        'Best Params': grid_search.best_params_
    })

    print(f"  Best CV Score: {grid_search.best_score_:.4f}")
    print(f"  Best Parameters: {grid_search.best_params_}\n")

## 10. Final Model Evaluation on Test Set

In [ ]:
# Evaluate all tuned models on test set
final_results = []

print("Evaluating models on test set...\n")

for model_name, model in best_models.items():
    # Predictions
    y_pred = model.predict(X_test)

    # Metrics
    r2 = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)

    final_results.append({
        'Model': model_name,
        'R2 Score': r2,
        'RMSE': rmse,
        'MAE': mae
    })

    print(f"{model_name}:")
    print(f"  R2 Score: {r2:.4f}")
    print(f"  RMSE: ${rmse:.2f}")
    print(f"  MAE: ${mae:.2f}\n")

In [ ]:
# Create final results dataframe
final_results_df = pd.DataFrame(final_results)
final_results_df = final_results_df.sort_values('R2 Score', ascending=False).reset_index(drop=True)

print("\n" + "=" * 60)
print("FINAL MODEL COMPARISON - TEST SET RESULTS")
print("=" * 60)
print(final_results_df.to_string(index=False))
print("=" * 60)

In [ ]:
# Visualize final results
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# R2 Score
axes[0].barh(final_results_df['Model'], final_results_df['R2 Score'], color='steelblue')
axes[0].set_xlabel('R2 Score')
axes[0].set_title('R2 Score Comparison')
axes[0].grid(axis='x', alpha=0.3)

# RMSE
axes[1].barh(final_results_df['Model'], final_results_df['RMSE'], color='coral')
axes[1].set_xlabel('RMSE ($)')
axes[1].set_title('RMSE Comparison (Lower is Better)')
axes[1].grid(axis='x', alpha=0.3)

# MAE
axes[2].barh(final_results_df['Model'], final_results_df['MAE'], color='lightgreen')
axes[2].set_xlabel('MAE ($)')
axes[2].set_title('MAE Comparison (Lower is Better)')
axes[2].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Identify best model
best_model_name = final_results_df.iloc[0]['Model']
best_model = best_models[best_model_name]

print(f"\n BEST MODEL: {best_model_name}")
print(f"R2 Score: {final_results_df.iloc[0]['R2 Score']:.4f}")
print(f"RMSE: ${final_results_df.iloc[0]['RMSE']:.2f}")
print(f"MAE: ${final_results_df.iloc[0]['MAE']:.2f}")

## 11. Prediction Analysis

In [ ]:
# Get predictions from best model
y_pred_best = best_model.predict(X_test)

# Actual vs Predicted plot
plt.figure(figsize=(10, 6))
plt.scatter(y_test, y_pred_best, alpha=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel('Actual Price ($)')
plt.ylabel('Predicted Price ($)')
plt.title(f'Actual vs Predicted Prices - {best_model_name}')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Residual plot
residuals = y_test - y_pred_best

plt.figure(figsize=(10, 6))
plt.scatter(y_pred_best, residuals, alpha=0.5)
plt.axhline(y=0, color='r', linestyle='--', lw=2)
plt.xlabel('Predicted Price ($)')
plt.ylabel('Residuals ($)')
plt.title(f'Residual Plot - {best_model_name}')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Error distribution
plt.figure(figsize=(10, 6))
plt.hist(residuals, bins=50, edgecolor='black')
plt.xlabel('Residuals ($)')
plt.ylabel('Frequency')
plt.title(f'Distribution of Residuals - {best_model_name}')
plt.axvline(x=0, color='r', linestyle='--', lw=2)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()